# Fix bugs in data merging of ESCO and O*NET
Felix Zaussinger | 13.04.2022

## Core Analysis Goal(s)
1. Fix bugs in occupation and skills metadata merging.
2. Gain confidence that the final datasets are valid.
3.

## Key Insight(s)
1.
2.
3.

In [172]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
# import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths(fn_config_path="paths_config.yml")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [173]:
from src import utils, stats_utils, plotting_utils
from src.data.esco import EscoDs

Configuration files

In [174]:
fn_config_path = "paths_config.yml"
fn_config_data = "data_config.yml"
fn_config_model = "model_config.yml"
fn_config_vis = "vis_config.yml"

Code ...

In [175]:
esco = EscoDs(fn_config_path=fn_config_path, fn_config_data=fn_config_data)

Problems with ONET-ESCO crosswalk

In [176]:
gt_esco = esco.read_greenness_task_based()

In [177]:
gt_esco.title_gtp.unique()

array(['Chief Sustainability Officers', 'Green Marketers',
       'Geothermal Production Managers', 'Biofuels Production Managers',
       'Biomass Production Managers',
       'Methane/Landfill Gas Collection System Operators',
       'Hydroelectric Production Managers',
       'Biofuels/Biodiesel Technology and Product Development Managers',
       'Water Resource Specialists', 'Wind Energy Operations Managers',
       'Wind Energy Project Managers',
       'Brownfield Redevelopment Specialists and Site Managers',
       'Energy Auditors', 'Sustainability Specialists',
       'Environmental Engineers', 'Water/Wastewater Engineers',
       'Fuel Cell Engineers', 'Energy Engineers', 'Wind Energy Engineers',
       'Solar Energy Systems Engineers',
       'Environmental Engineering Technicians', 'Fuel Cell Technicians',
       'Soil and Water Conservationists', 'Climate Change Analysts',
       'Environmental Restoration Planners', 'Industrial Ecologists',
       'Environmental Economis

63 ONET greening occupations dont have a match to ESCO based on current crosswalk

In [178]:
bsel = gt_esco.loc[:, ["preferred_label"]].isna().values
df_no_match = gt_esco.loc[bsel]
df_no_match

,onet_code,title_gtp,occupation_type,n_new_green_tasks_gtp,n_existing_green_tasks_gtp,n_non_green_tasks_gtp,share_green_gtp,title_vona2018,share_green_vona2018,total_spec_tasks_vona2018,green_spec_tasks_vona2018,id,concept_uri,preferred_label,isco_level_4,onet_occupation
1,11-2011.01,Green Marketers,New Green N&E,16,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11-3051.02,Geothermal Production Managers,New Green N&E,17,0,0,1.000000,Geothermal Production Managers,1.000000,17.0,17.0,NaN,NaN,NaN,NaN,NaN
3,11-3051.03,Biofuels Production Managers,New Green N&E,14,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11-3051.04,Biomass Production Managers,New Green N&E,18,0,0,1.000000,Biomass Power Plant Managers,1.000000,18.0,18.0,NaN,NaN,NaN,NaN,NaN
5,11-3051.05,Methane/Landfill Gas Collection System Operators,New Green N&E,21,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,11-3051.06,Hydroelectric Production Managers,New Green N&E,19,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,11-9041.01,Biofuels/Biodiesel Technology and Product Deve...,New Green N&E,19,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,11-9121.02,Water Resource Specialists,New Green N&E,21,0,0,1.000000,Water Resource Specialists,1.000000,21.0,21.0,NaN,NaN,NaN,NaN,NaN
9,11-9199.09,Wind Energy Operations Managers,New Green N&E,16,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,11-9199.10,Wind Energy Project Managers,New Green N&E,15,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [179]:
gt_esco.columns

Index(['onet_code', 'title_gtp', 'occupation_type', 'n_new_green_tasks_gtp',
       'n_existing_green_tasks_gtp', 'n_non_green_tasks_gtp',
       'share_green_gtp', 'title_vona2018', 'share_green_vona2018',
       'total_spec_tasks_vona2018', 'green_spec_tasks_vona2018', 'id',
       'concept_uri', 'preferred_label', 'isco_level_4', 'onet_occupation'],
      dtype='object')

Summary: number of ESCO occupations matched to a single greening ONET occupation

In [180]:
grouping_cols = ["onet_code", "title_gtp", "occupation_type", "share_green_gtp"]

match_summary = gt_esco.groupby(grouping_cols)["preferred_label"].count().sort_values(ascending=False)
match_summary.rename("n_matches_esco", inplace=True)

match_summary.to_csv(
    os.path.join(useful_paths.results_dir, "validation", "onet_esco_crosswalk", "onet_to_esco_match_summary.csv"),
    sep=";"
)

match_summary

onet_code   title_gtp                                                                                            occupation_type        share_green_gtp
13-1022.00  Wholesale and Retail Buyers, Except Farm Products                                                    Green Enhanced Skills  0.300000           41
17-2141.00  Mechanical Engineers                                                                                 Green Enhanced Skills  0.285714           36
51-9012.00  Separating, Filtering, Clarifying, Precipitating, and Still Machine Setters, Operators, and Tenders  Green Enhanced Skills  0.047619           29
51-9061.00  Inspectors, Testers, Sorters, Samplers, and Weighers                                                 Green Enhanced Skills  0.062500           28
27-3022.00  Reporters and Correspondents                                                                         Green Enhanced Skills  0.035714           18
                                                          

In [181]:
df_no_match.title_gtp.unique().shape

(63,)

In [183]:
onet_code = "11-3051.00"
esco.crosswalk_onet_esco_mcc_full.query("onet_code == '{}'".format(onet_code))

,id,concept_uri,preferred_label,isco_level_4,onet_code,onet_occupation
45,45,http://data.europa.eu/esco/occupation/02f26b2f...,footwear quality manager,1321,11-3051.00,industrial production managers
50,50,http://data.europa.eu/esco/occupation/0368c4d4...,metal production manager,1321,11-3051.00,industrial production managers
69,69,http://data.europa.eu/esco/occupation/0552ff45...,foundry manager,1219,11-3051.00,industrial production managers
161,161,http://data.europa.eu/esco/occupation/0d21a30d...,chemical production manager,1321,11-3051.00,industrial production managers
384,384,http://data.europa.eu/esco/occupation/1f8967ce...,leather goods quality manager,1321,11-3051.00,industrial production managers
467,467,http://data.europa.eu/esco/occupation/262d4e45...,leather wet processing department manager,1321,11-3051.00,industrial production managers
599,599,http://data.europa.eu/esco/occupation/316309e4...,leather production manager,1321,11-3051.00,industrial production managers
714,714,http://data.europa.eu/esco/occupation/3afd6af4...,clothing operations manager,1321,11-3051.00,industrial production managers
821,821,http://data.europa.eu/esco/occupation/45517bc3...,metallurgical manager,1321,11-3051.00,industrial production managers
1009,1009,http://data.europa.eu/esco/occupation/546ec458...,water treatment plant manager,1219,11-3051.00,industrial production managers
